# XGBoost 6 Models: Train → FGI Weight Recalc → Retrain
- **3M 테이블**: LAG1/2/3 사용 (LAG6 제외)
- **6M 테이블**: LAG1/2/3/6 사용
- NULL 처리: ZERO(0), MEAN(평균), REMOVE(행삭제)

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.inspection import permutation_importance
import xgboost as xgb
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

session = get_active_session()

FEATURES_3M = [
    'FGI_LAG1', 'FGI_LAG2', 'FGI_LAG3',
    'MONTH_SIN', 'MONTH_COS',
    'CONTRACT_COUNT_LAG1', 'CONTRACT_COUNT_LAG2', 'CONTRACT_COUNT_LAG3',
    'CHANGE_MEME_PRICE_RATE_LAG1', 'CHANGE_MEME_PRICE_RATE_LAG2', 'CHANGE_MEME_PRICE_RATE_LAG3'
]
FEATURES_6M = [
    'FGI_LAG1', 'FGI_LAG2', 'FGI_LAG3', 'FGI_LAG6',
    'MONTH_SIN', 'MONTH_COS',
    'CONTRACT_COUNT_LAG1', 'CONTRACT_COUNT_LAG2', 'CONTRACT_COUNT_LAG3', 'CONTRACT_COUNT_LAG6',
    'CHANGE_MEME_PRICE_RATE_LAG1', 'CHANGE_MEME_PRICE_RATE_LAG2', 'CHANGE_MEME_PRICE_RATE_LAG3', 'CHANGE_MEME_PRICE_RATE_LAG6'
]
TARGET = 'CONTRACT_COUNT'

FGI_COMPONENTS = ['CHANGE_MEME_PRICE_RATE','REVERSE_JEONSE_PER_MEME','RATE_GAP','MOVEMENT_RATE','PRICE_MOMENTUM_3M','PRICE_VOLATILITY_6M']
FGI_LABELS = ['매매변동률','전세가율역방향','괴리율','순이동인구','3개월모멘텀','6개월표준편차']

TABLES = {
    'ZERO_3M':  ('MY_DB.PUBLIC.FEATURE_NN_ZERO_3M', FEATURES_3M),
    'ZERO_6M':  ('MY_DB.PUBLIC.FEATURE_NN_ZERO_6M', FEATURES_6M),
    'MEAN_3M':  ('MY_DB.PUBLIC.FEATURE_NN_MEAN_3M', FEATURES_3M),
    'MEAN_6M':  ('MY_DB.PUBLIC.FEATURE_NN_MEAN_6M', FEATURES_6M),
    'REMOVE_3M':('MY_DB.PUBLIC.FEATURE_NN_REMOVE_3M', FEATURES_3M),
    'REMOVE_6M':('MY_DB.PUBLIC.FEATURE_NN_REMOVE_6M', FEATURES_6M),
}
print('Config ready. 6 models to train.')

In [ ]:
def load_data(table_name):
    df = session.sql(f'SELECT * FROM {table_name}').to_pandas()
    return df.dropna()

def train_xgb(X_train, y_train):
    model = xgb.XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42)
    model.fit(X_train, y_train)
    return model

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        'MAE': mean_absolute_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'R2': r2_score(y_test, y_pred)
    }

def calc_fgi_weights(df):
    df_clean = df[FGI_COMPONENTS + [TARGET]].dropna()
    scaler = MinMaxScaler()
    X_s = pd.DataFrame(scaler.fit_transform(df_clean[FGI_COMPONENTS]), columns=FGI_COMPONENTS)
    y = df_clean[TARGET].values
    reg = LinearRegression().fit(X_s, y)
    w_reg = np.abs(reg.coef_); w_reg = w_reg / w_reg.sum()
    gb = GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42).fit(X_s, y)
    perm = permutation_importance(gb, X_s, y, n_repeats=10, random_state=42)
    w_perm = np.abs(perm.importances_mean); w_perm = w_perm / w_perm.sum()
    final = (w_reg + w_perm) / 2
    final = final / final.sum()
    return np.round(final, 2)

def recalc_fgi(df, weights):
    df = df.copy()
    df['FEAR_GREED_INDEX'] = (
        df['CHANGE_MEME_PRICE_RATE'] * weights[0] +
        df['REVERSE_JEONSE_PER_MEME'] * weights[1] +
        df['RATE_GAP'] * weights[2] +
        df['MOVEMENT_RATE'] * weights[3] +
        df['PRICE_MOMENTUM_3M'] * weights[4] +
        df['PRICE_VOLATILITY_6M'] * weights[5]
    ) * 100
    for lag in [1, 2, 3, 6]:
        col = f'FGI_LAG{lag}'
        if col in df.columns:
            df[col] = df.groupby(['STATE','CITY'])['FEAR_GREED_INDEX'].shift(lag)
    return df.dropna()

print('Functions defined.')

In [ ]:
results_v1 = {}
results_v2 = {}
new_weights_all = {}

for name, (table, features) in TABLES.items():
    print(f'\n{"="*60}')
    print(f'  [{name}] Processing...')
    print(f'{"="*60}')
    
    df = load_data(table)
    X = df[features]; y = df[TARGET]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model_v1 = train_xgb(X_tr, y_tr)
    m1 = evaluate(model_v1, X_te, y_te)
    results_v1[name] = m1
    print(f'  V1 -> MAE: {m1["MAE"]:.2f}, RMSE: {m1["RMSE"]:.2f}, R2: {m1["R2"]:.4f}')
    
    weights = calc_fgi_weights(df)
    adj = 1.0 - weights.sum()
    weights[np.argmax(weights)] += adj
    new_weights_all[name] = weights
    print(f'  New Weights: {dict(zip(FGI_LABELS, weights))}')
    
    df_v2 = recalc_fgi(df, weights)
    X2 = df_v2[features]; y2 = df_v2[TARGET]
    X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X2, y2, test_size=0.2, random_state=42)
    
    model_v2 = train_xgb(X_tr2, y_tr2)
    m2 = evaluate(model_v2, X_te2, y_te2)
    results_v2[name] = m2
    print(f'  V2 -> MAE: {m2["MAE"]:.2f}, RMSE: {m2["RMSE"]:.2f}, R2: {m2["R2"]:.4f}')
    print(f'  R2 Change: {m2["R2"]-m1["R2"]:+.4f}')

print('\n All 6 models trained.')

In [ ]:
print(f'{"Model":<14} {"V1 MAE":>8} {"V1 RMSE":>9} {"V1 R2":>8} {"V2 MAE":>8} {"V2 RMSE":>9} {"V2 R2":>8} {"R2 Diff":>9}')
print('-' * 80)
for name in TABLES:
    v1 = results_v1[name]; v2 = results_v2[name]
    print(f'{name:<14} {v1["MAE"]:>8.2f} {v1["RMSE"]:>9.2f} {v1["R2"]:>8.4f} {v2["MAE"]:>8.2f} {v2["RMSE"]:>9.2f} {v2["R2"]:>8.4f} {v2["R2"]-v1["R2"]:>+9.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
models = list(TABLES.keys())
metrics = ['MAE', 'RMSE', 'R2']

for ax, metric in zip(axes, metrics):
    v1_vals = [results_v1[m][metric] for m in models]
    v2_vals = [results_v2[m][metric] for m in models]
    x = np.arange(len(models))
    w = 0.35
    ax.bar(x - w/2, v1_vals, w, label='V1 (before)', color='#FF6B6B', alpha=0.8)
    ax.bar(x + w/2, v2_vals, w, label='V2 (after)', color='#4ECDC4', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=45, ha='right', fontsize=8)
    ax.set_title(metric, fontsize=14)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('V1 vs V2 Performance Comparison', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for idx, (name, weights) in enumerate(new_weights_all.items()):
    ax = axes[idx // 3][idx % 3]
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD']
    bars = ax.barh(range(len(FGI_LABELS)), weights, color=colors)
    ax.set_yticks(range(len(FGI_LABELS)))
    ax.set_yticklabels(FGI_LABELS, fontsize=7)
    ax.set_title(f'{name}', fontsize=11)
    ax.set_xlim(0, max(weights) * 1.4)
    for bar, val in zip(bars, weights):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.2f}', va='center', fontsize=8)

plt.suptitle('FGI Weights by Model', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
print('\n' + '=' * 60)
print('  BEST MODEL SELECTION')
print('=' * 60)

best_name = max(results_v2, key=lambda k: results_v2[k]['R2'])
best = results_v2[best_name]
best_w = new_weights_all[best_name]

print(f'\n  Best Model: {best_name}')
print(f'  R2: {best["R2"]:.4f}, MAE: {best["MAE"]:.2f}, RMSE: {best["RMSE"]:.2f}')
print(f'\n  Optimal FGI Formula:')
print(f'  FGI = ', end='')
terms = [f'({l} x {w:.2f})' for l, w in zip(FGI_LABELS, best_w)]
print(' + '.join(terms) + ' x 100')

In [ ]:
best_table = 'MY_DB.PUBLIC.FEATURE_NN_ZERO_6M'
df_best = session.sql(f'SELECT * FROM {best_table}').to_pandas().dropna()

X_best = df_best[FEATURES_6M]
y_best = df_best[TARGET]

best_model = xgb.XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42)
best_model.fit(X_best, y_best)

imp = best_model.feature_importances_
imp_df = pd.DataFrame({'Feature': FEATURES_6M, 'Importance': imp}).sort_values('Importance', ascending=False)

print('=== ZERO_6M Feature Importance (CONTRACT_COUNT 예측) ===')
print(f'{"Rank":<5} {"Feature":<35} {"Importance":>10}')
print('-' * 55)
for i, (_, row) in enumerate(imp_df.iterrows(), 1):
    bar = '*' * int(row['Importance'] * 50)
    print(f'{i:<5} {row["Feature"]:<35} {row["Importance"]:>10.4f}  {bar}')

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#FF6B6B' if i < 3 else '#4ECDC4' for i in range(len(imp_df))]
ax.barh(range(len(imp_df)), imp_df['Importance'].values, color=colors)
ax.set_yticks(range(len(imp_df)))
ax.set_yticklabels(imp_df['Feature'].values, fontsize=9)
ax.set_xlabel('Feature Importance', fontsize=12)
ax.set_title('Top Features for CONTRACT_COUNT Prediction (ZERO_6M)', fontsize=13)
ax.invert_yaxis()
for i, v in enumerate(imp_df['Importance'].values):
    ax.text(v + 0.005, i, f'{v:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

# Test Data Validation (2025)

In [ ]:
df_test = session.sql('SELECT * FROM MY_DB.PUBLIC.FEATURE_TEST').to_pandas().dropna()
print(f'Test rows (after dropna): {len(df_test)}')

X_test = df_test[FEATURES_6M]
y_test = df_test[TARGET]

y_pred = best_model.predict(X_test)

mae_test = mean_absolute_error(y_test, y_pred)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
r2_test = r2_score(y_test, y_pred)

print(f'\n=== Test Performance (2025 Data) ===')
print(f'MAE:  {mae_test:.2f}')
print(f'RMSE: {rmse_test:.2f}')
print(f'R2:   {r2_test:.4f}')
print(f'\n=== Train Performance (2023-2024) ===')
print(f'R2:   {results_v1["ZERO_6M"]["R2"]:.4f}')
print(f'\nR2 Gap (Train-Test): {results_v1["ZERO_6M"]["R2"] - r2_test:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.scatter(y_test, y_pred, alpha=0.3, s=10, color='#4ECDC4')
max_val = max(y_test.max(), y_pred.max())
ax.plot([0, max_val], [0, max_val], 'r--', lw=1)
ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
ax.set_title(f'Actual vs Predicted (R2={r2_test:.4f})')
ax.set_aspect('equal')

ax = axes[1]
residuals = y_test.values - y_pred
ax.hist(residuals, bins=50, color='#FF6B6B', alpha=0.7, edgecolor='black', linewidth=0.5)
ax.axvline(0, color='black', linestyle='--')
ax.set_xlabel('Residual (Actual - Predicted)'); ax.set_ylabel('Count')
ax.set_title(f'Residual Distribution (Mean={np.mean(residuals):.1f})')

ax = axes[2]
monthly = df_test.groupby('YEAR_MONTH').agg(actual=(TARGET, 'mean')).reset_index()
df_test_copy = df_test.copy()
df_test_copy['PRED'] = y_pred
monthly_pred = df_test_copy.groupby('YEAR_MONTH').agg(predicted=('PRED', 'mean')).reset_index()
monthly = monthly.merge(monthly_pred, on='YEAR_MONTH')
ax.plot(range(len(monthly)), monthly['actual'], 'o-', label='Actual', color='#FF6B6B')
ax.plot(range(len(monthly)), monthly['predicted'], 's--', label='Predicted', color='#4ECDC4')
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels([str(d)[:7] for d in monthly['YEAR_MONTH']], rotation=45, fontsize=8)
ax.set_ylabel('Avg CONTRACT_COUNT'); ax.set_title('Monthly Avg: Actual vs Predicted')
ax.legend()

plt.tight_layout()
plt.show()

# Hyperparameter Tuning + Test Validation
GridSearchCV로 최적 파라미터 탐색 → 6개 모델 재학습 → 2025 테스트 검증

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.03, 0.05, 0.1],
    'min_child_weight': [3, 5, 10],
    'reg_alpha': [0, 0.1, 1.0],
    'reg_lambda': [1.0, 3.0, 5.0],
    'subsample': [0.8],
    'colsample_bytree': [0.8]
}

df_tune = session.sql('SELECT * FROM MY_DB.PUBLIC.FEATURE_NN_ZERO_6M').to_pandas().dropna()
X_tune = df_tune[FEATURES_6M]
y_tune = df_tune[TARGET]

print('Starting GridSearchCV (this may take a few minutes)...')
gs = GridSearchCV(
    xgb.XGBRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=0
)
gs.fit(X_tune, y_tune)

print(f'\nBest Params: {gs.best_params_}')
print(f'Best CV R2:  {gs.best_score_:.4f}')
best_params = gs.best_params_

In [ ]:
df_test_all = session.sql('SELECT * FROM MY_DB.PUBLIC.FEATURE_TEST').to_pandas().dropna()

results_tuned = {}

print(f'{"Model":<14} {"Train R2":>9} {"Test R2":>9} {"Test MAE":>9} {"Test RMSE":>10} {"Gap":>8}')
print('-' * 65)

for name, (table, features) in TABLES.items():
    df_tr = session.sql(f'SELECT * FROM {table}').to_pandas().dropna()
    X_tr = df_tr[features]; y_tr = df_tr[TARGET]
    
    model_t = xgb.XGBRegressor(**best_params, random_state=42)
    model_t.fit(X_tr, y_tr)
    
    y_tr_pred = model_t.predict(X_tr)
    r2_train = r2_score(y_tr, y_tr_pred)
    
    test_features = [f for f in features if f in df_test_all.columns]
    df_te = df_test_all[test_features + [TARGET]].dropna()
    X_te = df_te[test_features]; y_te = df_te[TARGET]
    
    y_te_pred = model_t.predict(X_te)
    r2_test = r2_score(y_te, y_te_pred)
    mae_test = mean_absolute_error(y_te, y_te_pred)
    rmse_test = np.sqrt(mean_squared_error(y_te, y_te_pred))
    
    results_tuned[name] = {'train_r2': r2_train, 'test_r2': r2_test, 'mae': mae_test, 'rmse': rmse_test, 'model': model_t}
    print(f'{name:<14} {r2_train:>9.4f} {r2_test:>9.4f} {mae_test:>9.2f} {rmse_test:>10.2f} {r2_train-r2_test:>8.4f}')

print(f'\nBest params used: {best_params}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
models_list = list(TABLES.keys())
x = np.arange(len(models_list))
w = 0.35

ax = axes[0]
train_r2 = [results_tuned[m]['train_r2'] for m in models_list]
test_r2 = [results_tuned[m]['test_r2'] for m in models_list]
ax.bar(x - w/2, train_r2, w, label='Train R2', color='#FF6B6B', alpha=0.8)
ax.bar(x + w/2, test_r2, w, label='Test R2', color='#4ECDC4', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(models_list, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('R2')
ax.set_title('Train vs Test R2 (Tuned)', fontsize=13)
ax.legend()
ax.grid(axis='y', alpha=0.3)
for i in range(len(models_list)):
    ax.text(i - w/2, train_r2[i] + 0.01, f'{train_r2[i]:.3f}', ha='center', fontsize=7)
    ax.text(i + w/2, test_r2[i] + 0.01, f'{test_r2[i]:.3f}', ha='center', fontsize=7)

ax = axes[1]
before_test = [results_v1[m]['R2'] if m in results_v1 else 0 for m in models_list]
ax.bar(x - w/2, [0.7424]*len(models_list), w, label='Before Tuning (ZERO_6M)', color='#DDA0DD', alpha=0.5)
ax.bar(x + w/2, test_r2, w, label='After Tuning', color='#4ECDC4', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(models_list, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Test R2')
ax.set_title('Test R2: Before vs After Tuning', fontsize=13)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
best_tuned_name = max(results_tuned, key=lambda k: results_tuned[k]['test_r2'])
bt = results_tuned[best_tuned_name]

print('=' * 60)
print('  FINAL BEST MODEL (Test Performance)')
print('=' * 60)
print(f'  Model:     {best_tuned_name}')
print(f'  Train R2:  {bt["train_r2"]:.4f}')
print(f'  Test R2:   {bt["test_r2"]:.4f}')
print(f'  Test MAE:  {bt["mae"]:.2f}')
print(f'  Test RMSE: {bt["rmse"]:.2f}')
print(f'  Gap:       {bt["train_r2"]-bt["test_r2"]:.4f}')
print(f'\n  Params: {best_params}')

In [ ]:
df_test_all = session.sql('SELECT * FROM MY_DB.PUBLIC.FEATURE_TEST').to_pandas().dropna()

full_results = {}

for name, (table, features) in TABLES.items():
    df_tr = session.sql(f'SELECT * FROM {table}').to_pandas().dropna()
    X_tr = df_tr[features]; y_tr = df_tr[TARGET]
    test_features = [f for f in features if f in df_test_all.columns]
    df_te = df_test_all[test_features + [TARGET]].dropna()
    X_te = df_te[test_features]; y_te = df_te[TARGET]

    model_before = xgb.XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42)
    model_before.fit(X_tr, y_tr)
    tr_pred_b = model_before.predict(X_tr)
    te_pred_b = model_before.predict(X_te)

    model_after = xgb.XGBRegressor(**best_params, random_state=42)
    model_after.fit(X_tr, y_tr)
    tr_pred_a = model_after.predict(X_tr)
    te_pred_a = model_after.predict(X_te)

    full_results[name] = {
        'b_tr_mae': mean_absolute_error(y_tr, tr_pred_b), 'b_tr_rmse': np.sqrt(mean_squared_error(y_tr, tr_pred_b)), 'b_tr_r2': r2_score(y_tr, tr_pred_b),
        'b_te_mae': mean_absolute_error(y_te, te_pred_b), 'b_te_rmse': np.sqrt(mean_squared_error(y_te, te_pred_b)), 'b_te_r2': r2_score(y_te, te_pred_b),
        'a_tr_mae': mean_absolute_error(y_tr, tr_pred_a), 'a_tr_rmse': np.sqrt(mean_squared_error(y_tr, tr_pred_a)), 'a_tr_r2': r2_score(y_tr, tr_pred_a),
        'a_te_mae': mean_absolute_error(y_te, te_pred_a), 'a_te_rmse': np.sqrt(mean_squared_error(y_te, te_pred_a)), 'a_te_r2': r2_score(y_te, te_pred_a),
    }

print(f'{"":<14} |{"--- Before Tuning (Train) ---":^30}|{"--- Before Tuning (Test) ---":^30}|{"--- After Tuning (Train) ---":^30}|{"--- After Tuning (Test) ---":^30}|')
print(f'{"Model":<14} |{"MAE":>8} {"RMSE":>8} {"R2":>8}   |{"MAE":>8} {"RMSE":>8} {"R2":>8}   |{"MAE":>8} {"RMSE":>8} {"R2":>8}   |{"MAE":>8} {"RMSE":>8} {"R2":>8}   |')
print('-' * 134)
for name in TABLES:
    r = full_results[name]
    print(f'{name:<14} |{r["b_tr_mae"]:>8.2f} {r["b_tr_rmse"]:>8.2f} {r["b_tr_r2"]:>8.4f}   |{r["b_te_mae"]:>8.2f} {r["b_te_rmse"]:>8.2f} {r["b_te_r2"]:>8.4f}   |{r["a_tr_mae"]:>8.2f} {r["a_tr_rmse"]:>8.2f} {r["a_tr_r2"]:>8.4f}   |{r["a_te_mae"]:>8.2f} {r["a_te_rmse"]:>8.2f} {r["a_te_r2"]:>8.4f}   |')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
metric_pairs = [
    ('b_tr_r2', 'a_tr_r2', 'Train R2'),
    ('b_te_r2', 'a_te_r2', 'Test R2'),
    ('b_tr_r2', 'b_te_r2', 'Before: Train vs Test R2'),
    ('b_tr_mae', 'a_tr_mae', 'Train MAE'),
    ('b_te_mae', 'a_te_mae', 'Test MAE'),
    ('a_tr_r2', 'a_te_r2', 'After: Train vs Test R2'),
]
label_pairs = [
    ('Before', 'After'),
    ('Before', 'After'),
    ('Train', 'Test'),
    ('Before', 'After'),
    ('Before', 'After'),
    ('Train', 'Test'),
]
color_pairs = [
    ('#FF6B6B', '#4ECDC4'),
    ('#FF6B6B', '#4ECDC4'),
    ('#FF6B6B', '#4ECDC4'),
    ('#FF6B6B', '#4ECDC4'),
    ('#FF6B6B', '#4ECDC4'),
    ('#FF6B6B', '#4ECDC4'),
]

models_list = list(TABLES.keys())
x = np.arange(len(models_list))
w = 0.35

for idx, (k1, k2, title) in enumerate(metric_pairs):
    ax = axes[idx // 3][idx % 3]
    v1 = [full_results[m][k1] for m in models_list]
    v2 = [full_results[m][k2] for m in models_list]
    l1, l2 = label_pairs[idx]
    ax.bar(x - w/2, v1, w, label=l1, color='#FF6B6B', alpha=0.8)
    ax.bar(x + w/2, v2, w, label=l2, color='#4ECDC4', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(models_list, rotation=45, ha='right', fontsize=8)
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)
    for i in range(len(models_list)):
        ax.text(i - w/2, v1[i], f'{v1[i]:.2f}', ha='center', va='bottom', fontsize=6)
        ax.text(i + w/2, v2[i], f'{v2[i]:.2f}', ha='center', va='bottom', fontsize=6)

plt.suptitle('Complete Performance Summary: 6 Models x Before/After Tuning', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
from snowflake.ml.registry import Registry

df_final = session.sql('SELECT * FROM MY_DB.PUBLIC.FEATURE_NN_REMOVE_6M').to_pandas().dropna()
FEATURES_6M = ['FGI_LAG1','FGI_LAG2','FGI_LAG3','FGI_LAG6','MONTH_SIN','MONTH_COS',
               'CONTRACT_COUNT_LAG1','CONTRACT_COUNT_LAG2','CONTRACT_COUNT_LAG3','CONTRACT_COUNT_LAG6',
               'CHANGE_MEME_PRICE_RATE_LAG1','CHANGE_MEME_PRICE_RATE_LAG2','CHANGE_MEME_PRICE_RATE_LAG3','CHANGE_MEME_PRICE_RATE_LAG6']

X_final = df_final[FEATURES_6M]
y_final = df_final['CONTRACT_COUNT']

final_model = xgb.XGBRegressor(
    colsample_bytree=0.8, learning_rate=0.1, max_depth=4,
    min_child_weight=10, n_estimators=100, reg_alpha=0, reg_lambda=5.0,
    subsample=0.8, random_state=42
)
final_model.fit(X_final, y_final)

reg = Registry(session=session, database_name='MY_DB', schema_name='PUBLIC')

mv = reg.log_model(
    model=final_model,
    model_name='xgboost_contract_prediction',
    version_name='v3_tuned_remove6m',
    sample_input_data=X_final.head(10),
    metrics={'test_r2': 0.7972, 'test_mae': 36.27, 'test_rmse': 63.89},
    comment='Tuned XGBoost REMOVE_6M: max_depth=4, min_child_weight=10, reg_lambda=5.0, n_estimators=100'
)

print(f'Model registered: {mv.model_name} / {mv.version_name}')
print(f'Test Metrics - R2: 0.7972, MAE: 36.27, RMSE: 63.89')